# 01 · One neuron draws a line

Video 001's claim, checked on play you record yourself.

**The question:** is Mario in the air? The game knows: byte `0x001D` is `1` (jumping) or `2` (falling off a ledge) when he's airborne. That byte is the **answer key**. The neuron never sees it; it only gets Mario's height, and has to guess.

**What 001 says** (`greenstone/docs/series/games/001.md`, chosen on sessions 1–2, tested on session 3):

| | right |
|---|---|
| always say "on the ground" | 54% |
| one neuron, one number (height), one cut | **94%** (the cut sits at the floor) |

Most of its mistakes are Mario standing on something higher than the floor: a pipe, a block, a step. And for a second question, *will he land within 10 frames?*, two numbers (how low he is, how fast he's falling) beat either alone: 91% against 84%, with a line that tilts.

The plan: record some random play → look at it → build the neuron → choose its setting on two sessions, test on a third → look at its mistakes → two numbers.

| address | what | units |
|---|---|---|
| `0x001D` | in the air? (**the answer key**) | `0` ground, `1` jumping, `2` falling, `3` flagpole |
| `0x00CE` | depth: how far down the screen he is | pixels; `176` standing on the floor |
| `0x009F` | vertical speed | signed whole pixels a frame; negative rising, positive falling |
| `0x000E` | player state | `08` = the player is playing |
| `0x00B5` | vertical screen page | `2` or more: he's fallen out of the bottom of the screen |

In [ ]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ipywidgets import interact, IntSlider, FloatSlider

import greenstone as gs

AIR, DEPTH, VSPEED, STATE, PAGE = 0x001D, 0x00CE, 0x009F, 0x000E, 0x00B5
DATA = Path("data")  # recordings are cached here (git ignores data/)


def show(picture, title=None):
    plt.figure(figsize=(4, 3.75))
    plt.imshow(picture)
    plt.axis("off")
    if title:
        plt.title(title, fontsize=9)
    plt.show()

## A random player

The same player 001 used (`greenstone/src/nes/gym/series/record.rs`): pick a button combination, mostly running right and jumping, hold it for 4 frames plus a random extra (on average 16 more), then pick again. It doesn't look at the screen at all.

When Mario dies, play starts over from the beginning of 1-1. Dying is player state `06` or `0B`, or falling out of the bottom of the screen (page 2 or more).

In [ ]:
COMBOS = [  # (buttons, how often)
    (gs.RIGHT, 0.30),
    (gs.RIGHT | gs.A, 0.25),
    (gs.RIGHT | gs.B, 0.10),
    (gs.RIGHT | gs.A | gs.B, 0.10),
    (0, 0.10),
    (gs.LEFT, 0.05),
    (gs.A, 0.10),
]


def random_player(seed, frames):
    """A button for every frame: combinations held for 4 + a geometric number of frames (mean 16)."""
    rng = np.random.default_rng(seed)
    buttons, weights = zip(*COMBOS)
    weights = np.array(weights) / sum(weights)
    track = []
    while len(track) < frames:
        held = buttons[rng.choice(len(buttons), p=weights)]
        track += [held] * (4 + rng.geometric(1 / 17) - 1)
    return np.array(track[:frames], np.uint8)


def died(ram):
    """Each frame: has Mario just died (dying, or fallen below the screen)?"""
    return np.isin(ram[:, STATE], (0x06, 0x0B)) | (ram[:, PAGE] >= 2)

`record` plays the buttons and keeps the memory after every frame. When Mario dies it keeps that frame, starts over, and goes on with the next buttons. It also keeps a saved moment every 600 frames (and at every restart), so any frame's picture can be made again later without storing 18,000 pictures.

In [ ]:
CHECK = 600


def record(seed, frames=18_000):
    # The cache's name says which player and how long, so a changed COMBOS records afresh.
    player = hashlib.sha1(repr(COMBOS).encode()).hexdigest()[:8]
    path = DATA / f"smb1-random-{player}-{seed}-{frames}.npz"
    if path.exists():
        z = np.load(path)
        return dict(ram=z["ram"], buttons=z["buttons"], restarts=list(z["restarts"]),
                    checkpoints=list(zip(z["check_frames"], z["check_states"])))
    nes = gs.boot("smb1")
    start = nes.save()
    buttons = random_player(seed, frames)
    rams, restarts, checkpoints = [], [], []
    f = 0
    while f < frames:
        checkpoints.append((f, np.frombuffer(nes.save(), np.uint8)))
        chunk = nes.play(buttons[f : f + CHECK], pictures=False)
        dead = np.flatnonzero(died(chunk))
        if len(dead):
            chunk = chunk[: dead[0] + 1]    # keep the frame he died on
            nes.load(start)
            restarts.append(f + dead[0])
        rams.append(chunk)
        f += len(chunk)
    ram = np.concatenate(rams)
    DATA.mkdir(exist_ok=True)
    frames_, states = zip(*checkpoints)
    np.savez_compressed(path, ram=ram, buttons=buttons, restarts=np.array(restarts, int),
                        check_frames=np.array(frames_), check_states=np.stack(states))
    return dict(ram=ram, buttons=buttons, restarts=restarts, checkpoints=checkpoints)


def picture(session, f):
    """The screen showing frame f's memory. The picture runs one frame behind the memory
    (00's lag exercise), so this is the picture drawn on frame f + 1."""
    cf, state = [c for c in session["checkpoints"] if c[0] <= f][-1]
    nes = gs.boot("smb1")
    nes.load(state.tobytes())
    _, pics = nes.play(session["buttons"][cf : f + 2], pictures=True)
    return pics[-1]

## Watch the data come out

One short session first (half a minute), to see the player and the bytes side by side. Watch the answer key flip as he jumps, and the depth sit at 176 on the floor.

In [ ]:
peek = record(seed=0, frames=1_800)
r = peek["ram"]

fig, ax = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
ax[0].plot(r[:, AIR]); ax[0].set_ylabel("in the air?\n(answer key)")
ax[1].plot(r[:, DEPTH]); ax[1].set_ylabel("depth (px)"); ax[1].invert_yaxis()
ax[2].plot(r[:, VSPEED].astype(np.int8)); ax[2].set_ylabel("vertical speed")
for a in ax:
    for x in peek["restarts"]:
        a.axvline(x, color="red", lw=0.8)
ax[2].set_xlabel("frame (red: died, started over)")
plt.show()

In [ ]:
def look(f):
    m = r[f]
    show(picture(peek, f),
         f"frame {f}: in the air? {m[AIR]}   depth {m[DEPTH]}   speed {np.int8(m[VSPEED])}   state {m[STATE]:02X}")

interact(look, f=IntSlider(0, 0, len(r) - 1));

## Three sessions

Five minutes each, as in 001. The first run takes a minute or two a session; after that they load from `data/`. Your random player isn't 001's (different random numbers), so the numbers won't match exactly, but they should land nearby: 001's sessions were in the air 43–47% of frames and restarted 34–41 times.

In [ ]:
sessions = [record(seed) for seed in (1, 2, 3)]
for i, s in enumerate(sessions, 1):
    air = np.isin(s["ram"][:, AIR], (1, 2))
    print(f"session {i}: {len(s['ram'])} frames, in the air {air.mean():.1%}, restarts {len(s['restarts'])}")

### Frames where the game moves Mario, not the player

While Mario dies, grows, shrinks, slides down the flagpole or goes down a pipe, the game is driving him, and the answer key can say odd things (dying sets "in the air" while he stands on the floor). 001 only counts frames where the player state `0x000E` is `08`, playing. Same here, and the frame he falls out of the screen is left out too (the depth byte wraps round to the top as he goes, so he'd look very high).

The neuron gets **height**, not depth, so that bigger means higher. Depth counts down the screen from somewhere above Mario's head; his feet are at depth + 32, and the screen is 240 tall, so `208 − depth` is how far his feet are above the bottom of the screen. On the floor that's **32**: the two rows of ground blocks. (In a pit it goes below zero for a few frames before he counts as dead; those are clipped to 0.)

In [ ]:
def frames_of(session):
    """Height, the answer key and vertical speed, for the frames where the player is playing."""
    ram = session["ram"]
    play = (ram[:, STATE] == 0x08) & ~died(ram)
    height = np.clip(208 - ram[play, DEPTH].astype(int), 0, None)
    air = np.isin(ram[play, AIR], (1, 2))
    return height, air, np.flatnonzero(play)


h12 = np.concatenate([frames_of(s)[0] for s in sessions[:2]])
air12 = np.concatenate([frames_of(s)[1] for s in sessions[:2]])
h3, air3, where3 = frames_of(sessions[2])
for i, s in enumerate(sessions, 1):
    print(f"session {i}: {len(s['ram']) - len(frames_of(s)[0])} frames left out")

## Look before building anything

Sessions 1 and 2 only (3 is for testing). How high is Mario, split by what the answer key says?

In [ ]:
bins = np.arange(0, h12.max() + 2) - 0.5
plt.figure(figsize=(9, 3.5))
plt.hist(h12[~air12], bins, alpha=0.6, label="on the ground")
plt.hist(h12[air12], bins, alpha=0.6, label="in the air")
plt.yscale("log"); plt.xlabel("height (px above the bottom of the screen)"); plt.ylabel("frames (log scale)")
plt.legend(); plt.show()
print(f"always say 'on the ground': {(~air12).mean():.1%} right on sessions 1–2")

Before reading on: where would *you* put a single cut?

## The neuron

One neuron with one input: multiply the input by a **weight**, add a **bias**, and say "yes" if the total is above zero.

$$\text{in the air} \iff w \cdot \text{height} + b > 0$$

**Exercise 1.** Write it. It should take an array of heights and return an array of `True`/`False`.

In [ ]:
def neuron(height, w, b):
    # Exercise 1
    raise NotImplementedError


def accuracy(guess, answer):
    return (guess == answer).mean()

Drag the dials. The dashed line is where the total crosses zero: the cut. What setting gets the most right? What does a negative weight do?

In [ ]:
def dials(w=1.0, b=-40.0):
    a = accuracy(neuron(h12, w, b), air12)
    plt.figure(figsize=(9, 3.5))
    plt.hist(h12[~air12], bins, alpha=0.6, label="on the ground")
    plt.hist(h12[air12], bins, alpha=0.6, label="in the air")
    if w != 0:
        plt.axvline(-b / w, color="k", ls="--", label=f"cut at {-b / w:.1f}")
    plt.yscale("log"); plt.xlabel("height"); plt.legend()
    plt.title(f"w = {w:.2f}, b = {b:.1f}: {a:.1%} right on sessions 1–2")
    plt.show()

interact(dials, w=FloatSlider(1.0, min=-2, max=2, step=0.05), b=FloatSlider(-40, min=-200, max=200, step=0.5));

## Choose on sessions 1–2, test on session 3

**Exercise 2.** Find the best setting by trying every cut on sessions 1–2 (with one input, a cut is all that matters: fix `w = 1` and try every `b`). Then score that setting once on session 3, which it has never seen. 001 says 94%, with the cut at the floor.

**Exercise 3.** Is it a fluke of which sessions you chose on? Choose the cut on each session alone. Do they agree? What does each score on the others?

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

## Its mistakes

Put your chosen setting in `W, B` and scrub through the frames of session 3 it gets wrong. 001 says most are Mario standing on something higher than the floor. Is that what you see? What are the rest?

In [ ]:
W, B = 1.0, -40.0  # your setting from exercise 2

wrong = np.flatnonzero(neuron(h3, W, B) != air3)
print(f"{len(wrong)} wrong of {len(h3)} ({1 - len(wrong) / len(h3):.1%} right)")


def mistake(i):
    j = wrong[i]
    f = where3[j]
    show(picture(sessions[2], f),
         f"frame {f}: height {h3[j]}, answer key {'air' if air3[j] else 'ground'}, neuron {'air' if not air3[j] else 'ground'}")

interact(mistake, i=IntSlider(0, 0, max(len(wrong) - 1, 0)));

**Exercise 4.** Count them: of the mistakes, how many are "on the ground" frames above the floor?

In [ ]:
# Exercise 4

## Two numbers: will he land soon?

A new question, for frames where Mario is in the air: will he be on the ground within the next 10 frames? Two inputs, both chosen so bigger means "more likely yes":

- **depth** (`0x00CE` as stored): bigger is lower, nearer the ground
- **falling speed** (`0x009F`, signed): bigger is falling faster; negative is still rising

Frames whose 10-frame look-ahead crosses a restart are left out (starting over isn't landing).

In [ ]:
SOON = 10


def landing(session):
    ram = session["ram"]
    air = np.isin(ram[:, AIR], (1, 2))
    play = (ram[:, STATE] == 0x08) & ~died(ram)
    crossed = np.zeros(len(ram), bool)
    for r_ in session["restarts"]:
        crossed[max(0, r_ - SOON + 1) : r_ + 1] = True
    keep = np.array([t for t in range(len(ram) - SOON) if air[t] and play[t] and not crossed[t]])
    lands = np.array([(~air[t + 1 : t + 1 + SOON]).any() for t in keep])
    x = np.stack([ram[keep, DEPTH].astype(int), ram[keep, VSPEED].astype(np.int8).astype(int)], axis=1)
    return x, lands, keep


x12, land12 = map(np.concatenate, zip(*[landing(s)[:2] for s in sessions[:2]]))
x3, land3, keep3 = landing(sessions[2])
print(f"{len(x12)} in-the-air frames on sessions 1–2; always say 'no': {(~land12).mean():.1%}")

The same neuron, two weights: yes if $w_1 \cdot \text{depth} + w_2 \cdot \text{falling speed} + b > 0$. The border is now a line. Each dot is a frame, jittered a little so stacked dots show.

**Exercise 5.** Make `neuron2(x, w1, w2, b)`, then use the dials: tilt the line, and see what one number alone gets (a weight of 0 ignores that input).

In [ ]:
def neuron2(x, w1, w2, b):
    # Exercise 5
    raise NotImplementedError


jitter = np.random.default_rng(0).uniform(-0.4, 0.4, x12.shape)


def dials2(w1=0.2, w2=1.0, b=-35.0):
    a = accuracy(neuron2(x12, w1, w2, b), land12)
    p = x12 + jitter
    plt.figure(figsize=(7, 5))
    plt.scatter(p[~land12, 0], p[~land12, 1], s=2, alpha=0.3, label="doesn't land within 10")
    plt.scatter(p[land12, 0], p[land12, 1], s=2, alpha=0.3, label="lands within 10")
    d = np.array([x12[:, 0].min(), x12[:, 0].max()])
    if w2 != 0:
        plt.plot(d, -(w1 * d + b) / w2, "k--")
    elif w1 != 0:
        plt.axvline(-b / w1, color="k", ls="--")
    plt.ylim(x12[:, 1].min() - 1, x12[:, 1].max() + 1)
    plt.xlabel("depth (bigger: lower)"); plt.ylabel("falling speed (bigger: falling faster)")
    plt.title(f"{a:.1%} right on sessions 1–2"); plt.legend(markerscale=5); plt.show()

interact(dials2, w1=FloatSlider(0.2, min=-1, max=1, step=0.01), w2=FloatSlider(1, min=-2, max=2, step=0.05),
         b=FloatSlider(-35, min=-200, max=200, step=0.5));

**Exercise 6.** Choose the best line on sessions 1–2 by search (try a grid of directions and cuts), and test it once on session 3. 001 says: depth alone 75%, speed alone 84%, both 91%, and the line tilts (lower *and* falling faster both point to "lands soon"). 001 also rounded it to whole numbers: depth + 5 × speed > 152 (that's 001's speed, the same byte), also 91%.

In [ ]:
# Exercise 6

## Play

1. **Its false alarms.** On the landing question, scrub the frames where the line says "lands soon" and he doesn't. 001 says nearly all are bounces off stomped enemies and falls into pits.
2. **A better player.** Change `COMBOS` (more running? fewer jumps?) and record new seeds. Does the best cut move? Does the score?
3. **Speed for the first question.** Add falling speed to the in-the-air neuron. 001 says one straight line doesn't get better: speed only helps when it *isn't zero*, which takes two cuts. Can you see why in a scatter?
4. **Another game.** `gs.boot("smb3")` and `gs.boot("castlevania")` work too. 001's answer keys and heights are in `greenstone/docs/series/games/001.md`.